### 1) DYNAMIC PTQ

In [1]:
import torch
import torch.nn as nn
import torch.nn. functional as F
import torch.optim as optim

from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import matplotlib.pyplot as plt
import numpy as np

In [2]:
x, y = make_moons(n_samples = 5000, noise = 0.5, random_state= 42)

In [3]:
x.shape, y.shape

((5000, 2), (5000,))

In [4]:
x = StandardScaler().fit_transform(x)
x = torch.tensor(x, dtype = torch.float32)
y = torch.tensor(y.reshape(-1, 1), dtype=torch.float32)

In [5]:
xTrain, xTest , yTrain, yTest = train_test_split(x, y, test_size=0.2, random_state= 42)
xTrain.shape, xTest.shape, yTrain.shape, yTest.shape

(torch.Size([4000, 2]),
 torch.Size([1000, 2]),
 torch.Size([4000, 1]),
 torch.Size([1000, 1]))

In [6]:
class Model(nn.Module):
  def __init__(self):
    super().__init__()
    self.net = nn.Sequential(
      nn.Linear(2, 128),
      nn.ReLU(),
      nn.Linear(128, 64),
      nn.ReLU(),
      nn.Linear(64, 64),
      nn.ReLU(),
      nn.Linear(64, 32),
      nn.ReLU(),
      nn.Linear(32, 16),
      nn.ReLU(),
      nn.Linear(16, 8),
      nn.ReLU(),
      nn.Linear(8, 1),
      nn.Sigmoid()
    )

  def forward(self, x):
    return self.net(x)

In [9]:
modelFP32 = Model()
modelFP32.parameters()

optimizer = optim.Adam(modelFP32.parameters(), lr=0.001)
criterion = nn.BCELoss()

for epoch in range(2000):
  modelFP32.train()
  optimizer.zero_grad()
  output = modelFP32(xTrain)
  loss = criterion(output, yTrain)
  loss.backward()
  optimizer.step()

  if epoch%100 == 0:
    print(f"Epoch {epoch} | Loss: {loss.item():.4f}")

Epoch 0 | Loss: 0.7083
Epoch 100 | Loss: 0.3962
Epoch 200 | Loss: 0.3890
Epoch 300 | Loss: 0.3869
Epoch 400 | Loss: 0.3855
Epoch 500 | Loss: 0.3841
Epoch 600 | Loss: 0.3832
Epoch 700 | Loss: 0.3826
Epoch 800 | Loss: 0.3815
Epoch 900 | Loss: 0.3825
Epoch 1000 | Loss: 0.3801
Epoch 1100 | Loss: 0.3798
Epoch 1200 | Loss: 0.3791
Epoch 1300 | Loss: 0.3785
Epoch 1400 | Loss: 0.3776
Epoch 1500 | Loss: 0.3784
Epoch 1600 | Loss: 0.3783
Epoch 1700 | Loss: 0.3751
Epoch 1800 | Loss: 0.3748
Epoch 1900 | Loss: 0.3738


In [15]:
def accuracy(model, x, y):
    model.eval()
    with torch.no_grad():
        preds = model(x)
        preds = (preds > 0.5).float()
        correct = (preds == y).float().mean()
        return correct.item()
accuracy(modelFP32, xTest, yTest)

0.8289999961853027

### Dynamic PTQ

In [14]:
from torch.quantization import quantize_dynamic
modelINT8 = quantize_dynamic(
    modelFP32,
    {nn.Linear},  # Which layer to quantize
    dtype=torch.qint8
)

print("INT 8 Quantized Accuracy", accuracy(modelINT8, xTest, yTest))

INT 8 Quantized Accuracy 0.8199999928474426


/tmp/ipython-input-3080307000.py:2: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  modelINT8 = quantize_dynamic(


In [16]:
modelINT8

Model(
  (net): Sequential(
    (0): DynamicQuantizedLinear(in_features=2, out_features=128, dtype=torch.qint8, qscheme=torch.per_tensor_affine)
    (1): ReLU()
    (2): DynamicQuantizedLinear(in_features=128, out_features=64, dtype=torch.qint8, qscheme=torch.per_tensor_affine)
    (3): ReLU()
    (4): DynamicQuantizedLinear(in_features=64, out_features=64, dtype=torch.qint8, qscheme=torch.per_tensor_affine)
    (5): ReLU()
    (6): DynamicQuantizedLinear(in_features=64, out_features=32, dtype=torch.qint8, qscheme=torch.per_tensor_affine)
    (7): ReLU()
    (8): DynamicQuantizedLinear(in_features=32, out_features=16, dtype=torch.qint8, qscheme=torch.per_tensor_affine)
    (9): ReLU()
    (10): DynamicQuantizedLinear(in_features=16, out_features=8, dtype=torch.qint8, qscheme=torch.per_tensor_affine)
    (11): ReLU()
    (12): DynamicQuantizedLinear(in_features=8, out_features=1, dtype=torch.qint8, qscheme=torch.per_tensor_affine)
    (13): Sigmoid()
  )
)

In [17]:
import os
torch.save(modelINT8.state_dict(), "modelINT8.pt")
torch.save(modelFP32.state_dict(), "modelFP32.pt")

In [20]:
print("FP32 Model Size (MB)", os.path.getsize("modelFP32.pt")/1e6)
print("INT8 Model Size (MB)", os.path.getsize("modelINT8.pt")/1e6)

FP32 Model Size (MB) 0.067489
INT8 Model Size (MB) 0.026387


In [29]:
def MannualQuantization(tensor, number_of_bits=8):
  qmin = -2**(number_of_bits - 1)
  qmax = 2**(number_of_bits - 1) - 1
  minVal, maxVal = tensor.min(), tensor.max()
  scale = (maxVal - minVal) / float(qmax - qmin + 1e-8)
  zeroPt = torch.round(-minVal / scale).to(torch.int32)
  qt = torch.clamp(torch.round(tensor / scale) + zeroPt, qmin, qmax).to(torch.int8)
  return qt, zeroPt, scale

In [27]:
def MannualDequantization(qt, scale, zeroPt):
  return (qt.float() - zeroPt) * scale

In [23]:
mannualModel = Model()
for name, param in mannualModel.named_parameters():
  print(name)
  print(param.shape)

net.0.weight
torch.Size([128, 2])
net.0.bias
torch.Size([128])
net.2.weight
torch.Size([64, 128])
net.2.bias
torch.Size([64])
net.4.weight
torch.Size([64, 64])
net.4.bias
torch.Size([64])
net.6.weight
torch.Size([32, 64])
net.6.bias
torch.Size([32])
net.8.weight
torch.Size([16, 32])
net.8.bias
torch.Size([16])
net.10.weight
torch.Size([8, 16])
net.10.bias
torch.Size([8])
net.12.weight
torch.Size([1, 8])
net.12.bias
torch.Size([1])


In [30]:
with torch.no_grad():
    for (name_fp, param_fp), (name_q, param_q) in zip(modelFP32.named_parameters(), mannualModel.named_parameters()):
        q_param, scale, zp = MannualQuantization(param_fp.data)
        dq_param = MannualDequantization(q_param, scale, zp)
        param_q.data.copy_(dq_param)

In [31]:
print("Mannual Quantized Accuracy", accuracy(mannualModel, xTest, yTest))

Mannual Quantized Accuracy 0.5239999890327454


### 2) STATIC PTQ

In [32]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np

X, y = make_moons(n_samples=500, noise=0.2, random_state=42)

X = StandardScaler().fit_transform(X)

X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y.reshape(-1, 1), dtype=torch.float32)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


class StaticModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(2, 16)
        self.fc2 = nn.Linear(16, 8)
        self.fc3 = nn.Linear(8, 1)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return torch.sigmoid(self.fc3(x))

model_fp32 = StaticModel()

optimizer = torch.optim.Adam(model_fp32.parameters(), lr=0.01)

loss_fn = nn.BCELoss()

for epoch in range(2000):
    model_fp32.train()
    optimizer.zero_grad()
    out = model_fp32(X_train)
    loss = loss_fn(out, y_train)
    loss.backward()
    optimizer.step()

def accuracy(model, X, y):
    model.eval()
    with torch.no_grad():
        preds = model(X)
        preds = (preds > 0.5).float()
        return (preds == y).float().mean().item()

print("\nFP32 Accuracy:", accuracy(model_fp32, X_test, y_test))


FP32 Accuracy: 0.9599999785423279


In [33]:
def quantize_tensor(t, num_bits=8):
    qmin = -2**(num_bits - 1) #-128
    qmax = 2**(num_bits - 1) - 1 #127
    min_val, max_val = t.min(), t.max()
    scale = (max_val - min_val) / (qmax - qmin + 1e-8)
    zp = torch.round(-min_val / scale).to(torch.int32)
    q_t = torch.clamp(torch.round(t / scale) + zp, qmin, qmax).to(torch.int8)
    return q_t, scale, zp

def dequantize_tensor(q_t, scale, zp):
    return (q_t.float() - zp) * scale

In [34]:
def get_activation_min_max(model, X_sample):
    act_ranges = {}
    hooks = []

    def register_hook(name):
        def hook_fn(module, input, output):
            act_min = output.min().item()
            act_max = output.max().item()
            act_ranges[name] = (act_min, act_max)
        return hook_fn

    for name, module in model.named_modules():
        if isinstance(module, nn.Linear):
            hooks.append(module.register_forward_hook(register_hook(name)))

    with torch.no_grad():
        model.eval()
        model(X_sample)

    for h in hooks:
        h.remove()

    return act_ranges


In [36]:
X_calib = X_train[:100]
act_ranges = get_activation_min_max(model_fp32, X_calib)
act_ranges

{'fc1': (-9.246996879577637, 5.3550238609313965),
 'fc2': (-26.7780704498291, 17.79416275024414),
 'fc3': (-121.07002258300781, 23.79676628112793)}

In [35]:
class QuantizedMLP(nn.Module):
    def __init__(self, fp_model, act_ranges):
        super().__init__()
        self.fc1 = nn.Linear(2, 16)
        self.fc2 = nn.Linear(16, 8)
        self.fc3 = nn.Linear(8, 1)

        # Quantize weights
        with torch.no_grad():
            for (name_fp, param_fp), (name_q, param_q) in zip(fp_model.named_parameters(), self.named_parameters()):
                q_param, scale, zp = quantize_tensor(param_fp.data)
                dq_param = dequantize_tensor(q_param, scale, zp)
                param_q.data.copy_(dq_param)

        # Store activation scales
        self.act_scales = {}
        for name, (amin, amax) in act_ranges.items():
            scale = (amax - amin) / 255.0
            zp = round(-amin / scale) if scale > 0 else 0
            self.act_scales[name] = (scale, zp)

    def quant_act(self, x, layer_name):
        scale, zp = self.act_scales[layer_name]
        q_x = torch.clamp(torch.round(x / scale) + zp, 0, 255).to(torch.uint8)
        dq_x = (q_x.float() - zp) * scale
        return dq_x

    def forward(self, x):
        x = F.relu(self.quant_act(self.fc1(x), 'fc1'))
        x = F.relu(self.quant_act(self.fc2(x), 'fc2'))
        return torch.sigmoid(self.fc3(x))

In [37]:
model_static_ptq = QuantizedMLP(model_fp32, act_ranges)
print("STATIC PTQ Accuracy:", accuracy(model_static_ptq, X_test, y_test))

STATIC PTQ Accuracy: 0.4300000071525574


### 3) QAT

In [38]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np
import torch.quantization

# Dataset
X, y = make_moons(n_samples=500, noise=0.2, random_state=42)
X = StandardScaler().fit_transform(X)
X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y.reshape(-1, 1), dtype=torch.float32)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

def accuracy(model, X, y):
    model.eval()
    with torch.no_grad():
        preds = model(X)
        preds = (preds > 0.5).float()
        return (preds == y).float().mean().item()

class QATMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.quant = torch.quantization.QuantStub()
        self.fc1 = nn.Linear(2, 16)
        self.fc2 = nn.Linear(16, 8)
        self.fc3 = nn.Linear(8, 1)
        self.dequant = torch.quantization.DeQuantStub()

    def forward(self, x):
        x = self.quant(x)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = torch.sigmoid(self.fc3(x))
        x = self.dequant(x)
        return x

In [42]:
model = QATMLP()
model.qconfig = torch.quantization.get_default_qat_qconfig('fbgemm')
torch.quantization.prepare_qat(model, inplace=True)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.BCELoss()

for epoch in range(2000):
    model.train()
    optimizer.zero_grad()
    out = model(X_train)
    loss = loss_fn(out, y_train)
    loss.backward()
    optimizer.step()
    if epoch % 100 == 0:
        print(f"Epoch {epoch} | Loss: {loss.item():.4f}")

/tmp/ipython-input-3572751405.py:3: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  torch.quantization.prepare_qat(model, inplace=True)


Epoch 0 | Loss: 0.6758
Epoch 100 | Loss: 0.5359
Epoch 200 | Loss: 0.4030
Epoch 300 | Loss: 0.2831
Epoch 400 | Loss: 0.2005
Epoch 500 | Loss: 0.1528
Epoch 600 | Loss: 0.1160
Epoch 700 | Loss: 0.0943
Epoch 800 | Loss: 0.0804
Epoch 900 | Loss: 0.0685
Epoch 1000 | Loss: 0.0641
Epoch 1100 | Loss: 0.0483
Epoch 1200 | Loss: 0.0474
Epoch 1300 | Loss: 0.0403
Epoch 1400 | Loss: 0.0395
Epoch 1500 | Loss: 0.0394
Epoch 1600 | Loss: 0.0352
Epoch 1700 | Loss: 0.0312
Epoch 1800 | Loss: 0.0335
Epoch 1900 | Loss: 0.0280


In [45]:
print("Accuracy before convert():", accuracy(model, X_test, y_test))

Accuracy before convert(): 0.9700000286102295


In [44]:
model_int8 = torch.quantization.convert(model.eval(), inplace=False)

/tmp/ipython-input-553789867.py:1: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  model_int8 = torch.quantization.convert(model.eval(), inplace=False)


In [46]:
print("QAT INT8 Accuracy:", accuracy(model_int8, X_test, y_test))

QAT INT8 Accuracy: 0.9700000286102295
